## 1. Statistical Validation — Product Group and Offer Response

The SQL analysis showed a substantial difference in offer-response rates between customers with one product and customers with two or more products.

A chi-square test of independence is used to test whether product group and offer-response status are statistically associated.

Null hypothesis (H₀): Product group and offer response are independent.

Alternative hypothesis (H₁): Product group and offer response are associated.

The test will be evaluated using the chi-square statistic and p-value.

In [42]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [33]:
import mysql.connector
from getpass import getpass

conn = mysql.connector.connect(
    host='localhost',
    user=input('MySQL username: '),
    password=getpass('MySQL password: '),
    database='banking_analysis'
)

print("Connected successfully")

MySQL username:  root
MySQL password:  ········


Connected successfully


In [60]:
query = """
SELECT
    num_products,
    responded_to_offer_clean,
    late_payments_last_year,
    churned_clean
FROM accounts;
"""

account_sql = pd.read_sql(query, conn)

/var/folders/mt/n8_vz8jx4p5dxpftk4rdzlw40000gp/T/ipykernel_31858/2932931862.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  account_sql = pd.read_sql(query, conn)


## Customer Relationship Depth Analysis

Customers are grouped into two categories based on the number of products they hold:

- 1 Product
- 2+ Products

The objective is to compare these two groups across monthly spending, transaction activity, offer response, and churn.

This analysis helps assess whether deeper customer relationships are associated with different levels of engagement, offer response, and churn.

In [48]:
product_group = np.where(
    account_sql['num_products']==1 , '1 Product', '2+ Products'
    )

product_group

array(['1 Product', '2+ Products', '2+ Products', ..., '1 Product',
       '2+ Products', '1 Product'], shape=(10996,), dtype='<U11')

In [47]:
response_table = pd.crosstab(
    product_group,
    account_sql['responded_to_offer_clean']
)

response_table


responded_to_offer_clean,0,1
row_0,,
1 Product,2758,1117
2+ Products,1218,5903


In [36]:
from scipy.stats import chi2_contingency

chi2, p_value, degrees_of_freedom, expected = chi2_contingency(
    response_table
)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", degrees_of_freedom)

Chi-square statistic: 3175.80772302541
p-value: 0.0
Degrees of freedom: 1


## 1.1 Chi-Square Test Results

The chi-square test produced a statistic of **3175.81** with **1 degree of freedom**.

The p-value was reported as **0.0**, indicating that the actual p-value is extremely small and below the level of numerical precision displayed by the software.

Since the p-value is far below the 0.05 significance level, the null hypothesis of independence is rejected.

There is strong statistical evidence of an association between product group and offer-response status in the current dataset.

This result supports the substantial difference in observed response rates between single-product and multi-product customers.

The test establishes statistical association, not causation.

## 2. Statistical Validation — Late Payments and Churn

The SQL analysis showed that customers with higher numbers of late payments have higher observed churn rates.

A chi-square test of independence is used to determine whether late-payment behaviour is statistically associated with customer churn.

To avoid a very small group for customers with five late payments, late payments are grouped into two categories:

`0-2 Late Payments` and `3+ Late Payments`.

Null hypothesis (H₀): Late-payment group and churn status are independent.

Alternative hypothesis (H₁): Late-payment group and churn status are associated.

In [58]:
late_payment_group = np.where(
    account_sql['late_payments_last_year'] <= 2 ,'0-2 Late Payments', '3+ Late Payments'
)

In [61]:
churn_table = pd.crosstab(
    late_payment_group,
    account_sql['churned_clean']
)

In [62]:
churn_table

churned_clean,0,1
row_0,,
0-2 Late Payments,8962,1314
3+ Late Payments,473,247


In [63]:
chi2, p_value,degree_of_freedom,epected = chi2_contingency(
    churn_table
)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", degrees_of_freedom)

Chi-square statistic: 254.01826580980114
p-value: 3.4550061674736333e-57
Degrees of freedom: 1


## 2.1 Chi-Square Test Results — Late Payments and Churn

The chi-square test produced a statistic of **254.02** with **1 degree of freedom**.

The p-value was **3.455 × 10⁻⁵⁷**, which is far below the 0.05 significance level.

Therefore, the null hypothesis of independence is rejected.

There is strong statistical evidence of an association between late-payment group and churn status in the current dataset.

The observed churn rate is substantially higher among customers with 3 or more late payments compared with customers with 0 to 2 late payments.

This result establishes statistical association and does not imply that late payments cause customer churn.